
# Análise de Funil e Teste A/A/B — Aplicativo de Alimentos

Este projeto analisa o comportamento dos usuários em um aplicativo de alimentos e avalia um experimento **A/A/B** relacionado a uma alteração de interface.

Os grupos **246** e **247** representam os grupos de controle, enquanto o grupo **248** representa o grupo experimental.

O projeto tem dois objetivos principais:

1. Analisar o funil de conversão entre as etapas principais do aplicativo.
2. Verificar, por meio de testes estatísticos, se a alteração de interface produziu diferenças significativas no comportamento dos usuários.

**Dashboard Tableau Public:**  
https://public.tableau.com/app/profile/giovani.vitor/viz/AnalisedeFunileTesteAAB/Dashboard-FunileTesteAAB?publish=yes



## 1. Importação das bibliotecas e carregamento dos dados

O conjunto de dados contém eventos registrados no aplicativo, identificando o tipo de evento, o usuário, o momento em que ocorreu e o grupo experimental ao qual o usuário pertence.


In [ ]:

# Bibliotecas utilizadas para manipulação, visualização e testes estatísticos
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.stats.proportion import proportions_ztest

# Carregamento do conjunto de dados
# Caso esteja executando localmente, ajuste o caminho conforme necessário.
data = pd.read_csv('/datasets/logs_exp_us.csv', sep='\t')

data.head()


In [ ]:

# Diagnóstico inicial
print('Dimensões iniciais:', data.shape)
print('\nValores ausentes por coluna:')
print(data.isna().sum())

print('\nRegistros duplicados:', data.duplicated().sum())
data.info()



### Conclusão da etapa

O conjunto original possui **244.126 registros**. Não foram identificados valores ausentes e foram encontrados **413 registros duplicados**.

Essas duplicidades serão removidas antes das análises para evitar distorções nas contagens de eventos e usuários.



## 2. Preparação e limpeza dos dados

Nesta etapa, os registros duplicados são removidos, as colunas recebem nomes mais descritivos e o timestamp é convertido para um formato de data e hora adequado para análise.


In [ ]:

# Remoção de registros duplicados
data = data.drop_duplicates().copy()

# Renomeação das colunas
data = data.rename(columns={
    'EventName': 'event_name',
    'DeviceIDHash': 'user_id',
    'EventTimestamp': 'event_timestamp',
    'ExpId': 'exp_id'
})

# Conversão do timestamp Unix para datetime
data['date_time'] = pd.to_datetime(data['event_timestamp'], unit='s')
data['date'] = data['date_time'].dt.date

print('Dimensões após remover duplicados:', data.shape)
data.head()



### Conclusão da etapa

Após a remoção de **413 registros duplicados**, o conjunto passou de **244.126 para 243.713 eventos**.

Também foram criadas as colunas `date_time` e `date`, facilitando a análise temporal do experimento.



## 3. Definição do período de análise

A distribuição temporal dos eventos é verificada para eliminar o período inicial com baixa cobertura de dados. A análise principal considera o período de **1º a 7 de agosto de 2019**.


In [ ]:

# Distribuição dos eventos ao longo do tempo
events_per_date = (
    data.groupby('date')
        .size()
        .reset_index(name='events')
)

plt.figure(figsize=(10, 4))
sns.barplot(data=events_per_date, x='date', y='events')
plt.title('Eventos por data')
plt.xlabel('Data')
plt.ylabel('Quantidade de eventos')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


In [ ]:

# Filtro do período considerado completo para análise
analysis = data[
    (data['date_time'] >= '2019-08-01') &
    (data['date_time'] < '2019-08-08')
].copy()

print('Eventos após o filtro:', len(analysis))
print('Usuários únicos:', analysis['user_id'].nunique())



### Conclusão da etapa

Após o filtro temporal, foram mantidos **240.887 eventos** de **7.534 usuários únicos**.

Esse recorte concentra o período com dados completos e será utilizado nas análises de funil e nos testes A/A/B.



## 4. Distribuição dos usuários entre os grupos

Antes de avaliar o experimento, é importante verificar se os usuários estão distribuídos de forma semelhante entre os três grupos.


In [ ]:

# Quantidade de usuários únicos por grupo experimental
users_by_group = (
    analysis.groupby('exp_id')['user_id']
            .nunique()
            .sort_index()
)

users_by_group



### Resultado

| Grupo | Papel no experimento | Usuários |
|---|---|---:|
| 246 | Controle A1 | 2.484 |
| 247 | Controle A2 | 2.513 |
| 248 | Experimental B | 2.537 |

Os três grupos possuem tamanhos próximos, o que favorece comparações entre eles.



## 5. Análise dos eventos e do funil de conversão

O funil principal do aplicativo é composto pelas seguintes etapas:

**Tela Principal → Ofertas → Carrinho → Pagamento Concluído**

O evento `Tutorial` é analisado separadamente, pois não representa uma etapa obrigatória do processo de compra.


In [ ]:

# Quantidade de usuários únicos e eventos por tipo de evento
event_summary = (
    analysis.groupby('event_name')
            .agg(
                unique_users=('user_id', 'nunique'),
                events=('user_id', 'size')
            )
            .sort_values('unique_users', ascending=False)
)

event_summary


In [ ]:

# Construção do funil principal
funnel_order = [
    'MainScreenAppear',
    'OffersScreenAppear',
    'CartScreenAppear',
    'PaymentScreenSuccessful'
]

funnel = (
    analysis[analysis['event_name'].isin(funnel_order)]
    .groupby('event_name')['user_id']
    .nunique()
    .reindex(funnel_order)
    .reset_index(name='users')
)

# Conversão em relação à etapa anterior e ao início do funil
funnel['conversion_previous_step'] = funnel['users'] / funnel['users'].shift(1)
funnel.loc[0, 'conversion_previous_step'] = 1

funnel['conversion_from_start'] = funnel['users'] / funnel.loc[0, 'users']

funnel


In [ ]:

# Visualização do funil
plt.figure(figsize=(9, 5))
sns.barplot(data=funnel, x='users', y='event_name')
plt.title('Funil de conversão')
plt.xlabel('Usuários únicos')
plt.ylabel('')
plt.tight_layout()
plt.show()



### Resultados do funil

| Etapa | Usuários | Conversão desde o início |
|---|---:|---:|
| Tela Principal | 7.419 | 100,0% |
| Ofertas | 4.593 | 61,9% |
| Carrinho | 3.734 | 50,3% |
| Pagamento Concluído | 3.539 | 47,7% |

A maior perda acontece entre **Tela Principal e Ofertas**, quando a conversão cai para aproximadamente **61,9%**.

A conversão total entre a primeira tela e o pagamento concluído é de aproximadamente **47,7%**.



## 6. Teste A/A — validação dos grupos de controle

Antes de comparar o grupo experimental com os grupos de controle, os grupos **246** e **247** são comparados entre si.

O objetivo do teste A/A é verificar se os dois grupos de controle apresentam comportamento estatisticamente equivalente.


In [ ]:

# Usuários únicos por evento e grupo
event_group_users = (
    analysis.groupby(['event_name', 'exp_id'])['user_id']
            .nunique()
            .unstack(fill_value=0)
)

group_sizes = analysis.groupby('exp_id')['user_id'].nunique()

def test_proportions(group_a, group_b, event):
    """Compara a proporção de usuários que realizaram um evento em dois grupos."""
    successes = [
        event_group_users.loc[event, group_a],
        event_group_users.loc[event, group_b]
    ]
    totals = [
        group_sizes.loc[group_a],
        group_sizes.loc[group_b]
    ]

    _, p_value = proportions_ztest(successes, totals)
    return p_value

events_to_test = [
    'Tutorial',
    'MainScreenAppear',
    'OffersScreenAppear',
    'CartScreenAppear',
    'PaymentScreenSuccessful'
]

aa_results = []
for event in events_to_test:
    p_value = test_proportions(246, 247, event)
    aa_results.append({
        'comparison': '246 vs 247',
        'test_type': 'A/A',
        'event': event,
        'p_value': p_value
    })

pd.DataFrame(aa_results)



### Resultado do teste A/A

Os cinco testes entre os grupos **246 e 247** apresentaram p-valores acima do nível de significância ajustado.

Isso indica que não foram identificadas diferenças estatisticamente significativas entre os dois grupos de controle, permitindo utilizar ambos como referência para a avaliação do grupo experimental.



## 7. Testes A/B

O grupo experimental **248** é comparado separadamente com cada grupo de controle e também com os grupos 246 e 247 combinados.


In [ ]:

def test_combined_controls(event):
    """Compara os controles 246+247 combinados com o grupo experimental 248."""
    control_successes = (
        event_group_users.loc[event, 246] +
        event_group_users.loc[event, 247]
    )
    control_total = group_sizes.loc[246] + group_sizes.loc[247]

    successes = [
        control_successes,
        event_group_users.loc[event, 248]
    ]
    totals = [
        control_total,
        group_sizes.loc[248]
    ]

    _, p_value = proportions_ztest(successes, totals)
    return p_value

test_results = []

# A/A
for event in events_to_test:
    test_results.append({
        'comparison': '246 vs 247',
        'test_type': 'A/A',
        'event': event,
        'p_value': test_proportions(246, 247, event)
    })

# A/B: 246 vs 248
for event in events_to_test:
    test_results.append({
        'comparison': '246 vs 248',
        'test_type': 'A/B',
        'event': event,
        'p_value': test_proportions(246, 248, event)
    })

# A/B: 247 vs 248
for event in events_to_test:
    test_results.append({
        'comparison': '247 vs 248',
        'test_type': 'A/B',
        'event': event,
        'p_value': test_proportions(247, 248, event)
    })

# A/B: controles combinados vs experimental
for event in events_to_test:
    test_results.append({
        'comparison': '246+247 vs 248',
        'test_type': 'A/B',
        'event': event,
        'p_value': test_combined_controls(event)
    })

results = pd.DataFrame(test_results)
results



## 8. Correção para múltiplos testes

Como foram realizados **20 testes estatísticos**, utiliza-se uma correção do nível de significância para reduzir a probabilidade de falsos positivos.

Com nível inicial de 5%:

\[
\alpha_{ajustado} = \frac{0,05}{20} = 0,0025
\]

Assim, um resultado é considerado estatisticamente significativo apenas quando **p < 0,0025**.


In [ ]:

alpha = 0.05
total_tests = 20
alpha_adjusted = alpha / total_tests

results['alpha_adjusted'] = alpha_adjusted
results['significant'] = results['p_value'] < alpha_adjusted

print('Nível de significância ajustado:', alpha_adjusted)
print('Testes significativos:', results['significant'].sum())

results.sort_values(['comparison', 'event'])



### Resumo dos p-valores obtidos

| Comparação | Tutorial | Tela Principal | Ofertas | Carrinho | Pagamento |
|---|---:|---:|---:|---:|---:|
| 246 vs 247 | 0,938 | 0,757 | 0,248 | 0,229 | 0,115 |
| 246 vs 248 | 0,826 | 0,295 | 0,208 | 0,078 | 0,212 |
| 247 vs 248 | 0,765 | 0,459 | 0,920 | 0,579 | 0,737 |
| 246+247 vs 248 | 0,765 | 0,294 | 0,434 | 0,182 | 0,600 |

Todos os p-valores ficaram acima de **0,0025**.



## 9. Conclusão geral e recomendações

A análise mostrou que o aplicativo possui **7.534 usuários** e **240.887 eventos** no período considerado.

O funil apresentou conversão total de aproximadamente **47,7%** entre a Tela Principal e o Pagamento Concluído. A principal perda ocorre na passagem da **Tela Principal para Ofertas**, etapa que merece maior atenção em futuras análises de experiência do usuário.

Os grupos do experimento apresentaram tamanhos semelhantes:

- Grupo 246: **2.484 usuários**
- Grupo 247: **2.513 usuários**
- Grupo 248: **2.537 usuários**

Foram realizados **20 testes A/A e A/B**, utilizando nível de significância ajustado de **0,0025**. Nenhum teste apresentou diferença estatisticamente significativa.

### Recomendações

- Não há evidência estatística suficiente para afirmar que a alteração de interface testada modificou o comportamento dos usuários.
- A mudança pode ser considerada neutra em relação às métricas analisadas no experimento.
- O principal ponto de oportunidade identificado está no início do funil, especialmente na transição entre a Tela Principal e a tela de Ofertas.
- Futuras iniciativas podem investigar elementos de navegação, comunicação e proposta de valor nessa etapa para reduzir a perda de usuários.



## Tecnologias utilizadas

- Python
- Pandas
- NumPy
- Matplotlib
- Seaborn
- Statsmodels
- Jupyter Notebook
- Tableau Public
